In [1]:
import sys
print(sys.version)

3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]


### CÀI ĐẶT HÀM

In [2]:
import numpy as np

def dinh_thuc(M, tol=1e-12): 
   
    M = np.array(M, dtype=float)  # tạo bản sao để không làm thay đổi ma trận gốc

    if M.ndim != 2 or M.shape[0] != M.shape[1]: # kiểm tra: ma trận là ma trận vuông nếu M là mảng 2 chiều và M có số hàng bằng số cột 
        raise ValueError("Định thức chỉ tính cho ma trận vuông.") 

    # khởi tạo
    n = M.shape[0]
    det = 1.0
    #chuyển ma trận M thành ma trận tam giác trên
    #-> det(M) = tích của các phần tử thuộc đường chéo chính
    
    for i in range(n):
        # Chọn phần tử chính (phần tử có giá trị max) theo cột
        p = i + np.argmax(np.abs(M[i:, i]))
        
        # nếu phần tử trụ quá nhỏ -> trụ xấp xỉ 0 -> suy biến
        if abs(M[p, i]) < tol:
            return 0.0

        # Đổi dòng thì định thức đổi dấu
        if p != i:
            M[[i, p]] = M[[p, i]]
            det = -det

        # tích các phần tử trụ (phần tử thuộc đường chéo chính)
        det *= M[i, i]

        # Khử các phần tử phía dưới phần tử chính (khử gauss)
        for j in range(i + 1, n):
            he_so = M[j, i] / M[i, i]
            M[j, i:] -= he_so * M[i, i:]

    return det


def conjugate_gradient(A, b, x0, eps) -> tuple[np.ndarray, int]:
    """
    A: Ma trận vuông, đối xứng xác định dương.
    b: Vectơ vế phải.
    x0: Vectơ nghiệm ban đầu.
    eps: Sai số cho phép, eps > 0.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()

    # Kiểm tra ma trận vuông
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A phải là ma trận vuông không rỗng.")

    # lưu cấp của A vào n
    n = A.shape[0]

    # Kiểm tra ma trận đối xứng
    if not np.allclose(A, A.T): # so sánh bằng giữ A vs A^T
        raise ValueError("A phải là ma trận đối xứng.")

    # Kiểm tra xác định dương 
    for k in range(1, n + 1):
        det_k = dinh_thuc(A[:k, :k]) # tính định thức con lấy ra từ k hàng và cột của A
        if det_k <= 0:
            raise ValueError("A không phải ma trận xác định dương ")

    # Khởi tạo phần dư (r) và hướng tìm kiếm (s)
    r = b - np.dot(A, x)
    s = r.copy()

    print("Nghiệm ban đầu x0:", x0)
    print("Phần dư ban đầu r0:", r)

    # kiểm tra nghiệm ban đầu 
    if np.sqrt(np.dot(r, r)) < eps:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0

    # vòng lặp
    for k in range(1, n + 1):

        # Tính alpha = s.r/s.As
        As = np.dot(A, s)
        mau = np.dot(s, As)
        if mau <= 0: # vì ma trận xác định dương nên mẫu phải dương
            raise ValueError("Mẫu số không hợp lệ.")

        alpha = np.dot(s, r) / mau

        # Cập nhật nghiệm, phần dư và chuẩn phần dư để xét điều kiện dừng
        x = x + alpha * s
        r_moi = b - np.dot(A, x)
        chuan_r = np.sqrt(np.sum(r_moi**2))

        # Hiển thị kết quả từng lần lặp
        print(f"\nLần lặp thứ {k}:")
        print("Nghiệm x =", [round(float(i), 6) for i in x])
        print("Phần dư r =", [round(float(i), 6) for i in r_moi])
        print("Chuẩn phần dư ||r|| =", round(float(chuan_r), 6))

        # Kiểm tra điều kiện dừng
        if chuan_r < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k
            
        # Tính beta
        beta = -np.dot(r_moi, As) / mau

        # Cập nhật hướng tìm kiếm
        s = r_moi + beta * s
        r = r_moi.copy()
        
    return x, n

In [3]:
A = np.array([
        [10, -1,  2,  0],
        [-1, 11, -1,  3],
        [2, -1,  10, -1],
        [0,  3, -1,  8],
], dtype=float)

b = np.array([6, 25, -11, 15], dtype=float)
x0 = np.zeros(4)

print("\n=== MA TRẬN CẤP 5 ====")
conjugate_gradient(A, b, x0, eps = 1e-5)


=== MA TRẬN CẤP 5 ====


Nghiệm ban đầu x0: [0. 0. 0. 0.]
Phần dư ban đầu r0: [  6.  25. -11.  15.]

Lần lặp thứ 1:
Nghiệm x = [0.471626, 1.965108, -0.864648, 1.179065]
Phần dư r = [4.978144, -0.546405, -0.152603, -1.192491]
Chuẩn phần dư ||r|| = 5.15032

Lần lặp thứ 2:
Nghiệm x = [0.996432, 1.976565, -0.909847, 1.097591]
Phần dư r = [-0.168064, 0.051594, -0.820239, -0.620272]
Chuẩn phần dư ||r|| = 1.043282

Lần lặp thứ 3:
Nghiệm x = [1.001525, 1.983269, -1.009858, 1.019696]
Phần dư r = [-0.012262, 0.116622, 0.0985, -0.117232]
Chuẩn phần dư ||r|| = 0.192864

Lần lặp thứ 4:
Nghiệm x = [1.0, 2.0, -1.0, 1.0]
Phần dư r = [0.0, 0.0, -0.0, 0.0]
Chuẩn phần dư ||r|| = 0.0
Thỏa mãn sau 4 lần lặp.


(array([ 1.,  2., -1.,  1.]), 4)

In [4]:
import pytest
import numpy as np
import io
import contextlib

A_MAU = [[4, 1], [1, 3]]
B_MAU = [1, 2]
X_MAU = [1 / 11, 7 / 11]


def bat_output(A, b, x0, eps):
    """Chạy conjugate_gradient, bắt phần in ra. Trả về (x, k, output)."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        x, k = conjugate_gradient(A, b, x0, eps)
    return x, k, buf.getvalue()


def lay_dong(out):
    return [d.strip() for d in out.splitlines() if d.strip()]


def tao_spd(rng, n):
    B = rng.standard_normal((n, n))
    return B @ B.T + n * np.eye(n)


# PHẦN 1: dinh_thuc 

# 1. Giá trị đúng với các ma trận tính tay được
def test_dinh_thuc_gia_tri_dung():
    assert dinh_thuc([[1, 2], [3, 4]]) == pytest.approx(-2.0)
    assert dinh_thuc([[5]]) == pytest.approx(5.0)
    assert dinh_thuc(np.diag([2, 3, 4])) == pytest.approx(24.0)
    assert dinh_thuc(np.eye(4)) == pytest.approx(1.0)


# 2. Phải đổi dòng -> định thức đổi dấu
def test_dinh_thuc_doi_dong():
    assert dinh_thuc([[0, 1], [1, 0]]) == pytest.approx(-1.0)


# 3. Suy biến trả về 0 và tham số tol
def test_dinh_thuc_suy_bien_va_tol():
    assert dinh_thuc([[1, 2], [2, 4]]) == 0.0
    assert dinh_thuc([[1, 2, 3], [0, 0, 0], [4, 5, 6]]) == 0.0
    assert dinh_thuc([[1e-3, 0], [0, 1e-3]], tol=1e-2) == 0.0
    assert dinh_thuc([[1e-3, 0], [0, 1e-3]], tol=1e-6) == pytest.approx(1e-6)


# 4. So với numpy trên ma trận ngẫu nhiên + không làm đổi ma trận gốc
def test_dinh_thuc_so_voi_numpy():
    rng = np.random.default_rng(0)
    for n in [2, 3, 4, 5, 8]:
        for _ in range(50):
            M = rng.standard_normal((n, n))
            goc = M.copy()
            assert dinh_thuc(M) == pytest.approx(np.linalg.det(M), rel=1e-6, abs=1e-8)
            assert np.array_equal(M, goc)


# 5. Báo lỗi với ma trận không vuông / không phải 2 chiều
def test_dinh_thuc_bao_loi():
    with pytest.raises(ValueError):
        dinh_thuc([[1, 2, 3], [4, 5, 6]])
    with pytest.raises(ValueError):
        dinh_thuc([1, 2, 3])


# PHẦN 2: conjugate_gradient

# 6. Ví dụ chuẩn: nghiệm đúng, tối đa n bước, kiểu dữ liệu trả về
def test_cg_vi_du_chuan():
    x, k, _ = bat_output(A_MAU, B_MAU, [0, 0], 1e-8)
    assert x == pytest.approx(X_MAU, abs=1e-8)
    assert k <= 2
    assert isinstance(x, np.ndarray) and isinstance(k, int)


# 7. Tính tay đúng 1 bước lặp và dừng sớm
def test_cg_mot_buoc_lap_tinh_tay():
    # alpha = 0.25, x1 = (0.25, 0.5), ||r1|| = 0.559 < 0.6 = eps
    x, k, _ = bat_output(A_MAU, B_MAU, [0, 0], 0.6)
    assert k == 1
    assert x == pytest.approx([0.25, 0.5])


# 8. Ma trận đặc biệt: đường chéo, đơn vị, 1x1
def test_cg_ma_tran_dac_biet():
    x, k, _ = bat_output(np.diag([1.0, 2.0, 3.0]), [1, 4, 9], [0, 0, 0], 1e-10)
    assert x == pytest.approx([1, 2, 3], abs=1e-9) and k <= 3

    b = [3.0, -1.0, 2.0]
    x, k, _ = bat_output(np.eye(3), b, [0, 0, 0], 1e-10)
    assert x == pytest.approx(b) and k == 1

    x, k, _ = bat_output([[2.0]], [4.0], [0.0], 1e-10)
    assert x == pytest.approx([2.0]) and k == 1


# 9. Dừng ngay từ đầu: x0 đã là nghiệm, hoặc eps rất lớn
def test_cg_dung_ngay_tu_dau():
    x, k, _ = bat_output(A_MAU, B_MAU, X_MAU, 1e-6)
    assert k == 0 and x == pytest.approx(X_MAU)

    _, k, _ = bat_output(A_MAU, B_MAU, [0, 0], 100.0)
    assert k == 0


# 10. Đầu vào: x0 khác 0, đầu vào list, không làm đổi x0
def test_cg_dau_vao():
    x0 = np.array([5.0, -3.0])
    goc = x0.copy()
    x, k, _ = bat_output(A_MAU, B_MAU, x0, 1e-8)
    assert x == pytest.approx(X_MAU, abs=1e-8) and k <= 2
    assert np.array_equal(x0, goc)

    x, _, _ = bat_output([[2, 0], [0, 2]], [2, 4], [0, 0], 1e-10)   # toàn list
    assert x == pytest.approx([1.0, 2.0])


# 11. So với np.linalg.solve trên ma trận SPD ngẫu nhiên
def test_cg_so_voi_numpy_ngau_nhien():
    rng = np.random.default_rng(0)
    for _ in range(200):
        n = int(rng.integers(2, 7))
        A = tao_spd(rng, n)
        b = rng.standard_normal(n)
        x, k, _ = bat_output(A, b, np.zeros(n), 1e-8)
        assert x == pytest.approx(np.linalg.solve(A, b), abs=1e-6)
        assert 1 <= k <= n
        assert np.linalg.norm(b - A @ x) < 1e-6


# 12. Báo lỗi hình dạng / đối xứng: không vuông, rỗng, 1 chiều, không đối xứng
def test_cg_bao_loi_hinh_dang():
    cac_ca = [
        ([[1, 2, 3], [4, 5, 6]], [1, 2], [0, 0], "vuông"),
        (np.empty((0, 0)), [], [], "vuông"),
        ([1, 2, 3], [1, 2, 3], [0, 0, 0], "vuông"),
        ([[2, 1], [0, 2]], [1, 1], [0, 0], "đối xứng"),
    ]
    for A, b, x0, thong_bao in cac_ca:
        with pytest.raises(ValueError, match=thong_bao):
            bat_output(A, b, x0, 1e-6)


# 13. Báo lỗi không xác định dương (và không in gì trước khi báo lỗi)
def test_cg_bao_loi_xac_dinh_duong():
    cac_ca = [
        [[1, 2], [2, 1]],      # đối xứng, định thức con cấp 2 < 0
        [[-1, 0], [0, -1]],    # xác định âm
        [[1, 1], [1, 1]],      # nửa xác định dương, suy biến
    ]
    for A in cac_ca:
        buf = io.StringIO()
        with pytest.raises(ValueError, match="xác định dương"):
            with contextlib.redirect_stdout(buf):
                conjugate_gradient(A, [1, 1], [0, 0], 1e-6)
        assert "Lần lặp thứ" not in buf.getvalue()
        assert "Nghiệm ban đầu x0" not in buf.getvalue()


# ===================== PHẦN 3: output (2 tiêu chí) =====================

# 14. Output đầy đủ của ví dụ chuẩn (so khớp từng dòng, đúng thứ tự)
def test_output_vi_du_chuan():
    _, _, out = bat_output(A_MAU, B_MAU, [0, 0], 1e-8)
    dong = lay_dong(out)
    assert dong[0] == "Nghiệm ban đầu x0: [0, 0]"
    assert dong[1].startswith("Phần dư ban đầu r0:")
    assert dong[2:] == [
        "Lần lặp thứ 1:",
        "Nghiệm x = [0.25, 0.5]",
        "Phần dư r = [-0.5, 0.25]",
        "Chuẩn phần dư ||r|| = 0.559017",
        "Lần lặp thứ 2:",
        "Nghiệm x = [0.090909, 0.636364]",
        "Phần dư r = [0.0, 0.0]",
        "Chuẩn phần dư ||r|| = 0.0",
        "Thỏa mãn sau 2 lần lặp.",
    ]
    assert "Lần lặp thứ 3" not in out


# 15. Output khi dừng sớm, khi x0 đã đạt, và khớp với giá trị trả về
def test_output_cac_truong_hop():
    _, _, out = bat_output(A_MAU, B_MAU, [0, 0], 0.6)
    assert "Thỏa mãn sau 1 lần lặp." in out
    assert "Lần lặp thứ 2" not in out

    _, _, out = bat_output(A_MAU, B_MAU, X_MAU, 1e-6)
    assert "Nghiệm ban đầu đã thỏa mãn điều kiện dừng." in out
    assert "Lần lặp thứ" not in out
    assert "Thỏa mãn sau" not in out

    rng = np.random.default_rng(1)
    for _ in range(100):
        n = int(rng.integers(2, 6))
        A = tao_spd(rng, n)
        b = rng.standard_normal(n)
        x, k, out = bat_output(A, b, np.zeros(n), 1e-3)
        dong = lay_dong(out)
        assert len([d for d in dong if d.startswith("Lần lặp thứ")]) == k
        in_ra = eval([d for d in dong if d.startswith("Nghiệm x =")][-1].split("=", 1)[1])
        assert in_ra == pytest.approx([round(float(v), 6) for v in x])
        chuan_cuoi = float([d for d in dong if d.startswith("Chuẩn phần dư")][-1].split("=")[1])
        assert chuan_cuoi < 1e-3


#  CHẠY TẤT CẢ TEST 

test_functions = [
    test_dinh_thuc_gia_tri_dung, test_dinh_thuc_doi_dong, test_dinh_thuc_suy_bien_va_tol,
    test_dinh_thuc_so_voi_numpy, test_dinh_thuc_bao_loi,
    test_cg_vi_du_chuan, test_cg_mot_buoc_lap_tinh_tay, test_cg_ma_tran_dac_biet,
    test_cg_dung_ngay_tu_dau, test_cg_dau_vao, test_cg_so_voi_numpy_ngau_nhien,
    test_cg_bao_loi_hinh_dang, test_cg_bao_loi_xac_dinh_duong,
    test_output_vi_du_chuan, test_output_cac_truong_hop,
]

so_pass = 0
for func in test_functions:
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            func()
        print(f"✔ {func.__name__} ---> PASSED")
        so_pass += 1
    except Exception as e:
        print(f"✘ {func.__name__} ---> FAILED: {type(e).__name__}: {e}")

print(f"\n{so_pass}/{len(test_functions)} test PASSED")
if so_pass == len(test_functions):
    print("pass")
assert so_pass == len(test_functions), "Có ca kiểm thử gradient liên hợp thất bại."


✔ test_dinh_thuc_gia_tri_dung ---> PASSED


✔ test_dinh_thuc_doi_dong ---> PASSED
✔ test_dinh_thuc_suy_bien_va_tol ---> PASSED
✔ test_dinh_thuc_so_voi_numpy ---> PASSED
✔ test_dinh_thuc_bao_loi ---> PASSED
✔ test_cg_vi_du_chuan ---> PASSED
✔ test_cg_mot_buoc_lap_tinh_tay ---> PASSED
✔ test_cg_ma_tran_dac_biet ---> PASSED
✔ test_cg_dung_ngay_tu_dau ---> PASSED
✔ test_cg_dau_vao ---> PASSED
✔ test_cg_so_voi_numpy_ngau_nhien ---> PASSED
✔ test_cg_bao_loi_hinh_dang ---> PASSED
✔ test_cg_bao_loi_xac_dinh_duong ---> PASSED
✔ test_output_vi_du_chuan ---> PASSED


✔ test_output_cac_truong_hop ---> PASSED

15/15 test PASSED
pass
